In [1]:
# https://www.kaggle.com/datasets/rajathmc/cornell-moviedialog-corpus

In [2]:
#!/bin/bash
!curl -L -o cornell-moviedialog-corpus.zip\
  https://www.kaggle.com/api/v1/datasets/download/rajathmc/cornell-moviedialog-corpus

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 9807k  100 9807k    0     0  15.0M      0 --:--:-- --:--:-- --:--:-- 15.0M


In [3]:
!unzip /content/cornell-moviedialog-corpus.zip

Archive:  /content/cornell-moviedialog-corpus.zip
  inflating: .DS_Store               
  inflating: README.txt              
  inflating: chameleons.pdf          
  inflating: movie_characters_metadata.txt  
  inflating: movie_conversations.txt  
  inflating: movie_lines.txt         
  inflating: movie_titles_metadata.txt  
  inflating: raw_script_urls.txt     


In [4]:
import re

In [5]:
lines = {}

with open('movie_lines.txt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.split(' +++$+++ ')
        if len(parts) == 5:
            lines[parts[0]] = parts[4].strip()

In [6]:
conversations = []

with open('movie_conversations.txt', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.split(' +++$+++ ')
        if len(parts) == 4:

            line_ids = eval(parts[3])
            conversations.append(line_ids)

In [7]:
questions = []
answers = []

for conv in conversations:
    for i in range(len(conv) - 1):
        questions.append(lines[conv[i]])
        answers.append(lines[conv[i+1]])

In [8]:
len(questions)

221616

In [9]:
len(answers)

221616

In [11]:
print(questions[0])
print()
print(answers[0])

Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.

Well, I thought we'd start with pronunciation, if that's okay with you.


In [12]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"i'm", "i am", text)
    text = re.sub(r"he's", "he is", text)
    text = re.sub(r"she's", "she is", text)
    text = re.sub(r"that's", "that is", text)
    text = re.sub(r"what's", "what is", text)
    text = re.sub(r"where's", "where is", text)
    text = re.sub(r"how's", "how is", text)
    text = re.sub(r"it's", "it is", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'ve", " have", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'d", " would", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"won't", " will not", text)
    text = re.sub(r"can't", " cannot", text)

    text = re.sub(r"[-()\"#/@;:<>{}`+=~|.!?,]", "", text)
    text = text.strip()

    return text

In [13]:
cleaned_questions = [clean_text(q) for q in questions]

cleaned_answers = [f"<SOS> {clean_text(a)} <EOS>" for a in answers]

In [14]:
print(cleaned_questions[0])
print()
print(cleaned_answers[0])

can we make this quick  roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad  again

<SOS> well i thought we would start with pronunciation if that is okay with you <EOS>


In [15]:
from collections import Counter
import numpy as np

In [16]:
word_counts = Counter()
for sentence in cleaned_questions + cleaned_answers:
    word_counts.update(sentence.split())

In [17]:
threshold = 5
vocab = [word for word, count in word_counts.items() if count >= threshold]

In [18]:
special_tokens = ['<PAD>', '<UNK>', '<SOS>', '<EOS>']

word2int = {word: i for i, word in enumerate(special_tokens + vocab)}
int2word = {i: word for word, i in word2int.items()}

In [19]:
len(word2int)

22538

In [20]:
def sentence_to_int(sentence, word2int):
    return [word2int.get(word, word2int['<UNK>']) for word in sentence.split()]

questions_int = [sentence_to_int(q, word2int) for q in cleaned_questions]
answers_int = [sentence_to_int(a, word2int) for a in cleaned_answers]

In [21]:
MAX_LEN = 20

def pad_sequence(sequences, max_len, pad_value):
    padded_seqs = []
    for seq in sequences:
        if len(seq) < max_len:
            padded_seqs.append(seq + [pad_value] * (max_len - len(seq)))
        else:
            padded_seqs.append(seq[:max_len])

    return np.array(padded_seqs)

In [22]:
pad_id = word2int['<PAD>']
questions_padded = pad_sequence(questions_int, MAX_LEN, pad_id)
answers_padded = pad_sequence(answers_int, MAX_LEN, pad_id)

In [23]:
print(cleaned_questions[0])
print()
print(questions_int[0])

can we make this quick  roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad  again

[4, 5, 6, 7, 8, 1, 1, 9, 10, 11, 12, 13, 14, 15, 1, 16, 17, 18, 19, 20, 1, 21]


In [24]:
print(len(questions_padded[0]))
print()
print(questions_padded[0])

20

[ 4  5  6  7  8  1  1  9 10 11 12 13 14 15  1 16 17 18 19 20]


In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random

In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'{device}')

cpu


In [27]:
input_data = torch.tensor(questions_padded, dtype=torch.long)
target_data = torch.tensor(answers_padded, dtype=torch.long)

In [28]:
BATCH_SIZE = 64

dataset = TensorDataset(input_data, target_data)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [29]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers=1):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=n_layers, batch_first=True)

    def forward(self, src):

        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)

        return hidden


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers=1):
        super(Decoder, self).__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden):

        input = input.unsqueeze(1)
        embedded = self.embedding(input)

        output, hidden = self.rnn(embedded, hidden)

        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):

        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        hidden = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden)
            outputs[:, t] = output

            top1 = output.argmax(1)


            input = trg[:, t] if random.random() < teacher_forcing_ratio else top1

        return outputs

In [30]:
INPUT_DIM = len(word2int)
OUTPUT_DIM = len(word2int)
EMB_DIM = 128
HIDDEN_DIM = 256
N_LAYERS = 1

enc = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM, N_LAYERS)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM, N_LAYERS)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [31]:
PAD_IDX = word2int['<PAD>']
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)


        output_dim = output.shape[-1]

        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

In [ ]:
import torch
import torch.nn.functional as F

def evaluate_beam_search(model, sentence, word2int, int2word, device, max_len=20, beam_width=3):

    model.eval()

    cleaned_src = clean_text(sentence)
    tokens = [word2int.get(word, word2int['<UNK>']) for word in cleaned_src.split()]

    if len(tokens) == 0:
        return "i do not know what to say"

    padded_tokens = tokens + [word2int['<PAD>']] * (max_len - len(tokens))
    src_tensor = torch.tensor(padded_tokens, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        hidden = model.encoder(src_tensor)


    sos_id = word2int['<SOS>']
    beams = [(0.0, [sos_id], hidden)]

    for _ in range(max_len):
        new_beams = []
        end_nodes_count = 0

        for score, seq, h in beams:
            if seq[-1] == word2int['<EOS>']:
                new_beams.append((score, seq, h))
                end_nodes_count += 1
                continue

            current_input = torch.tensor([seq[-1]], dtype=torch.long).to(device)

            with torch.no_grad():
                output, next_hidden = model.decoder(current_input, h)

            log_probs = F.log_softmax(output, dim=1).squeeze(0)

            topk_probs, topk_ids = torch.topk(log_probs, beam_width)

            for i in range(beam_width):
                next_score = score + topk_probs[i].item()
                next_seq = seq + [topk_ids[i].item()]
                new_beams.append((next_score, next_seq, next_hidden))

        new_beams = sorted(new_beams, key=lambda x: x[0], reverse=True)
        beams = new_beams[:beam_width]

        if end_nodes_count == beam_width:
            break

    best_seq = beams[0][1]

    output_words = []
    for token_id in best_seq:
        word = int2word[token_id]
        if word in ['<SOS>', '<EOS>', '<PAD>']:
            continue
        output_words.append(word)

    return " ".join(output_words)